# 01 — Explore Barnard's Star

Owner: B. Scratch only; never imported by the app. Anything that works moves into `backend/`.

Goals: list exposures covering Barnard's Star, check how many epochs exist per detector, open one cutout and look at its HDUs (IMAGE, FLAGS, VARIANCE, ZODI, PSF, WCS-WAVE).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from astropy.io import fits
from pyvo.dal import TAPService
from backend.catalogs import propagate

# Propagate J2000 position to ~2026 (approximate values, verify on SIMBAD)
ra, dec = propagate(269.452, 4.693, -801.551, 10362.394, 2000.0, 2026.0)
ra, dec

In [ ]:
tap = TAPService('https://irsa.ipac.caltech.edu/TAP')
q = f"""
SELECT a.uri, p.time_bounds_lower, p.energy_bandpassname
FROM spherex.artifact a JOIN spherex.plane p ON a.planeid = p.planeid
WHERE CONTAINS(POINT('ICRS', {ra}, {dec}), p.poly) = 1
"""
rows = tap.run_sync(q).to_table()
len(rows)

In [ ]:
# How many epochs per detector?
from backend.fits_processor import group_epochs
rows['epoch'] = group_epochs(list(rows['time_bounds_lower']))
rows.group_by(['energy_bandpassname', 'epoch']).groups.keys

In [ ]:
url = f"{rows['uri'][0]}?center={ra},{dec}&size=0.05"
hdul = fits.open(url)
hdul.info()